In [1]:
# ================================================================
#  LCCN-R2 - THRESHOLD SENSITIVITY + ABLATION MACRO-F1  [CPU, ~3-5 h]
#  Part A: rebuilds the full cascade (no transliteration) at
#    theta in {2,3,4}, delta in {1,2,3}, chi-square alpha in {.01,.05},
#    one knob at a time, reporting tokens changed, merges, and blocks.
#  Part B: builds the five constraint-ablation variants of Table VII
#    as full-corpus arms.
#  Part C: trains Logistic Regression and SVM (trainer v1.4,
#    byte-identical from the NB2 output) on every arm, adding the
#    macro-F1 column the ablation table lacks.
#  Add Input: paper1-nb1-lccn-final (output), paper1-nb2-lccn-ml
#             (output, for trainer.py), final-emotion-dataset, gold-ds.
#  Accelerator None. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library')
print("python", sys.version.split()[0])


+ pip install -q regex emoji wordfreq indic-nlp-library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 7.1 MB/s eta 0:00:00
python 3.12.13


In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE (v5.3, selected explicitly)
# ================================================================
import shutil, importlib, sys, glob, re, os
for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)
cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
assert cand, "attach the paper1-nb1-lccn-final output"
chosen = None
for p in cand:
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"',
                  open(p, encoding="utf-8", errors="replace").read())
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None: chosen = p
assert chosen, "no v5.3 pipeline.py among the inputs"
shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3" and FP == "6a536dd437ba85b0"


  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True


In [3]:
# ================================================================
#  CELL 2 - SHARED SETUP (replicates NB1 CELL 2; frozen split asserted)
# ================================================================
import importlib
importlib.reload(pipeline)
from pipeline import *
import pandas as pd, numpy as np
from collections import Counter

df, ID, TXT, LAB, RATERS = load_corpus()
norm_map, F2E, LEX = build_keel()
try:
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _n = IndicNormalizerFactory().get_normalizer("kn")
    _n.normalize("\u0c95"); pipeline._indic_norm = _n; print("IndicNLP: FULL")
except Exception:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    pipeline._indic_norm = _F(); print("IndicNLP: FALLBACK(NFC)")

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))
SPLIT, TEST_HASH = make_split(df, ID, LAB)
assert TEST_HASH == "e2d6e1f7d564cace", f"split drift: {TEST_HASH}"
train = df[df[ID].isin(SPLIT["train"])]
evidence = LabelEvidence(train, TXT, LAB)
TRAIN_FREQ = Counter()
for t in train.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ
print(f"train vocab {len(TRAIN_FREQ):,} | hash {TEST_HASH}")


corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
IndicNLP: FULL
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
train vocab 42,985 | hash e2d6e1f7d564cace


In [4]:
# ================================================================
#  CELL 3 - PART A: THRESHOLD SENSITIVITY (one knob at a time)
#  theta: length condition inside the veto (default 3)
#  delta: edit-distance cap of the orthographic guard (default 2)
#  alpha: chi-square level of the distributional/cluster tests (.01)
# ================================================================
from tqdm.auto import tqdm; tqdm.pandas()

def veto_theta(F2E, lex, theta):
    def veto(w, c, prev=None):
        if lex is not None:
            a, b = lex.emotion_in_context(w, prev), lex.emotion_in_context(c, prev)
        else:
            a, b = F2E.get(w), F2E.get(c)
        if a and b and len(a) == 1 and len(b) == 1 and a != b: return "emotion"
        if (w in pipeline.NEG) != (c in pipeline.NEG): return "negation"
        if abs(len(w) - len(c)) > theta: return "length"
        return None
    return veto

CRIT = {"05": {1: 3.84, 2: 5.99, 3: 7.81, 4: 9.49, 5: 11.07, 6: 12.59},
        "001": {1: 10.83, 2: 13.82, 3: 16.27, 4: 18.47, 5: 20.52, 6: 22.46}}
class EvA:
    def __init__(self, ev, table): self._ev, self._t = ev, table
    def conflict(self, f1, f2, threshold=None):
        G, dof = self._ev.g_test(f1, f2)
        return G > self._t.get(dof, list(self._t.values())[0] + 1.9*dof)
    def __getattr__(self, k): return getattr(self._ev, k)

_ORTHO = pipeline.orthographically_plausible
SETTINGS = [("base_t3_d2_a01", 3, 2, "01"),
            ("theta2",         2, 2, "01"),
            ("theta4",         4, 2, "01"),
            ("delta1",         3, 1, "01"),
            ("delta3",         3, 3, "01"),
            ("alpha05",        3, 2, "05"),
            ("alpha001",       3, 2, "001")]
sens_rows = []
for name, th, dl, al in SETTINGS:
    pipeline.orthographically_plausible = (
        lambda v, c, _d=dl: _ORTHO(v, c, max_edit=_d))
    ev = EvA(evidence, CRIT[al]) if al in CRIT else evidence
    o = LCCN(norm_map, ev, veto_theta(F2E, LEX, th), translit=None, lex=LEX)
    col = f"text_sens_{name}"
    df[col] = df[TXT].progress_apply(o.normalize)
    pipeline.orthographically_plausible = _ORTHO
    ch = tot = 0
    for a, b in zip(df["text_raw"], df[col]):
        ra, rb = a.lower().split(), b.lower().split()
        tot += len(ra); ch += sum(1 for u, v in zip(ra, rb) if u != v)
    st = o.stats
    blocked = {k: v for k, v in st.items() if k.startswith("blocked")}
    print(f"{name:<16} tokens_changed {100*ch/tot:5.2f}%  "
          f"merged {st['merged']:>6,}  {blocked}")
    sens_rows.append({"setting": name, "theta": th, "delta": dl,
                      "alpha": "0." + al, "tokens_changed_pct": round(100*ch/tot, 2),
                      "merged": st["merged"], **blocked})
pd.DataFrame(sens_rows).to_csv("/kaggle/working/lccn_r14_sensitivity.csv", index=False)


  0%|          | 0/40000 [00:00<?, ?it/s]

base_t3_d2_a01   tokens_changed 14.17%  merged  5,977  {'blocked_orthographic': 4062, 'blocked_gtest': 259, 'blocked_veto_negation': 44, 'blocked_veto_emotion': 114, 'blocked_cluster': 35, 'blocked_veto_length': 3}


  0%|          | 0/40000 [00:00<?, ?it/s]

theta2           tokens_changed 14.14%  merged  5,970  {'blocked_orthographic': 4062, 'blocked_gtest': 259, 'blocked_veto_negation': 44, 'blocked_veto_length': 24, 'blocked_veto_emotion': 114, 'blocked_cluster': 35}


  0%|          | 0/40000 [00:00<?, ?it/s]

...

delta3           tokens_changed 14.18%  merged  6,039  {'blocked_orthographic': 3990, 'blocked_gtest': 259, 'blocked_veto_negation': 44, 'blocked_veto_emotion': 114, 'blocked_cluster': 35, 'blocked_veto_length': 3}


  0%|          | 0/40000 [00:00<?, ?it/s]

alpha05          tokens_changed 13.63%  merged  5,845  {'blocked_orthographic': 4062, 'blocked_gtest': 604, 'blocked_veto_negation': 44, 'blocked_veto_emotion': 114, 'blocked_cluster': 71, 'blocked_veto_length': 3}


  0%|          | 0/40000 [00:00<?, ?it/s]

alpha001         tokens_changed 15.67%  merged  6,025  {'blocked_orthographic': 4062, 'blocked_gtest': 134, 'blocked_veto_negation': 44, 'blocked_cluster': 13, 'blocked_veto_emotion': 114, 'blocked_veto_length': 3}


In [5]:
# ================================================================
#  CELL 4 - PART B: THE FIVE ABLATION VARIANTS AS FULL-CORPUS ARMS
#  Same step order as the certified lccn_ablation() of pipeline v5.3.
# ================================================================
veto = make_veto(F2E, LEX)
STEPS = [
    ("ab1_none",    dict(use_veto=False, use_gtest=False, use_cluster=False, use_ortho=False)),
    ("ab2_vetoes",  dict(use_veto=True,  use_gtest=False, use_cluster=False, use_ortho=False)),
    ("ab3_ortho",   dict(use_veto=True,  use_gtest=False, use_cluster=False, use_ortho=True)),
    ("ab4_gtest",   dict(use_veto=True,  use_gtest=True,  use_cluster=False, use_ortho=True)),
    ("ab5_cluster", dict(use_veto=True,  use_gtest=True,  use_cluster=True,  use_ortho=True)),
]
for name, kw in STEPS:
    o = LCCN(norm_map, evidence, veto, translit=None, lex=LEX, **kw)
    df[f"text_{name}"] = df[TXT].progress_apply(o.normalize)
    st = o.stats
    print(f"{name:<12} merged {st['merged']:>6,}  "
          f"{ {k: v for k, v in st.items() if k.startswith('blocked')} }")


  0%|          | 0/40000 [00:00<?, ?it/s]

ab1_none     merged  8,131  {}


  0%|          | 0/40000 [00:00<?, ?it/s]

ab2_vetoes   merged  8,058  {'blocked_veto_negation': 49, 'blocked_veto_emotion': 197, 'blocked_veto_length': 5}


  0%|          | 0/40000 [00:00<?, ?it/s]

ab3_ortho    merged  6,072  {'blocked_orthographic': 4062, 'blocked_veto_negation': 44, 'blocked_veto_emotion': 114, 'blocked_veto_length': 3}


  0%|          | 0/40000 [00:00<?, ?it/s]

ab4_gtest    merged  5,987  {'blocked_orthographic': 4062, 'blocked_gtest': 259, 'blocked_veto_negation': 44, 'blocked_veto_emotion': 114, 'blocked_veto_length': 3}


  0%|          | 0/40000 [00:00<?, ?it/s]

ab5_cluster  merged  5,977  {'blocked_orthographic': 4062, 'blocked_gtest': 259, 'blocked_veto_negation': 44, 'blocked_veto_emotion': 114, 'blocked_cluster': 35, 'blocked_veto_length': 3}


In [6]:
# ================================================================
#  CELL 5 - PART C: CLASSICAL MACRO-F1 (trainer v1.4, byte-identical)
#  LR and SVM per arm; raw and lccn columns from the NB1 splits are
#  included as anchors. Numbers transcribe only from the committed log.
# ================================================================
import shutil, glob, os, sys, importlib
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
      if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
assert tc, "trainer.py not found - attach the paper1-nb2-lccn-ml output"
shutil.copy(tc[0], "/kaggle/working/trainer.py")
sys.modules.pop("trainer", None)
import trainer
importlib.reload(trainer)
from trainer import *
assert trainer.TRAINER_VERSION == "1.4", trainer.TRAINER_VERSION
trainer.fingerprint()

sd = [os.path.dirname(p) for p in
      glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
assert sd, "attach the paper1-nb1-lccn-final output (splits)"
tr, va, te, HAVE = load_splits(sd[0])

new_cols = ([f"text_sens_{n}" for n, *_ in SETTINGS]
            + [f"text_{n}" for n, _ in STEPS])
lut = {c: dict(zip(df[ID].astype(str), df[c])) for c in new_cols}
for f in (tr, va, te):
    ids = f["id"].astype(str)
    for c in new_cols:
        f[c.lower()] = ids.map(lut[c])
        assert f[c.lower()].notna().all(), f"id join failed for {c}"

ARMS = [c[5:] for c in tr.columns if c.startswith("text_")
        and (c[5:].startswith("sens_") or c[5:].startswith("ab")
             or c[5:] in ("raw", "lccn"))]
print("arms:", ARMS)
res = run_grid(tr, va, te, ARMS, ["Logistic Regression", "SVM"],
               tag="lccn_r14_15", resume=False)
print("\n=== macro-F1 (LR, SVM, and their mean) ===")
r_lr, r_svm = res.iloc[0], res.iloc[1]
print(f"{'arm':<18}{'LR':>8}{'SVM':>8}{'mean':>8}")
for a in ARMS:
    m = (float(r_lr[a]) + float(r_svm[a])) / 2
    print(f"{a:<18}{float(r_lr[a]):>8.4f}{float(r_svm[a]):>8.4f}{m:>8.4f}")
print("\nDONE - lccn_r14_sensitivity.csv + results_lccn_r14_15.csv written.")
print("Deltas below classical run-to-run noise are reported as no effect.")


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']
arms: ['raw', 'lccn', 'sens_base_t3_d2_a01', 'sens_theta2', 'sens_theta4', 'sens_delta1', 'sens_delta3', 'sens_alpha05', 'sens_alpha0